# XAI benchmark v2 su Colab

1. **Runtime solo CPU** (gratuito): passi 1 e 2, i test su modelli finti. Se falliscono, ci si ferma qui.
2. **Runtime GPU**: passo 1, passo 3 (stima dei tempi), poi passo 4, che fa tutto in parallelo.

I risultati vanno su Google Drive. Se Colab si disconnette, rilanciando la stessa cella il lavoro riprende da dove era arrivato, e il risultato è identico a quello di un run senza interruzioni.

## 1. Codice e dipendenze

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf /content/xai
!git clone -q --branch v2 https://github.com/nicolobachiorri/XAI_Benchmark_for_Sentiment_Analysis.git /content/xai
%cd /content/xai
!git log --oneline -1

In [ ]:
!pip install -q lime pytest
import os, torch, transformers
print(torch.__version__, transformers.__version__, 'GPU' if torch.cuda.is_available() else 'CPU', os.cpu_count(), 'CPU')

## 2. Test (CPU, pochi minuti, niente download)

In [ ]:
!python -m pytest -q tests

## 3. Stima dei tempi sull'hardware in uso (GPU)

Scarica i cinque modelli e cronometra ogni metodo su tre recensioni di lunghezza tipica. Il piano (passo 4 con `--dry-run`) mostra come verrà diviso il lavoro tra GPU e CPU.

In [ ]:
!python report.py --estimate
!python parallel_run.py --out-dir /content/drive/MyDrive/xai_v2 --dry-run

## 4. Run completo, tutto in parallelo

Sulla GPU girano insieme due processi con LIME, SHAP, Input×Gradient, Rollout e LRP; sulle CPU libere girano i processi di Attention Flow. Una riga di avanzamento ogni 5 minuti; i log dei singoli processi sono in `xai_v2/logs/`. Alla fine stampa le tre tabelle.

In [ ]:
!python parallel_run.py --out-dir /content/drive/MyDrive/xai_v2

### Variante: due runtime insieme

Se il runtime GPU ha poche CPU, Attention Flow diventa la parte più lunga. Si può aprire una seconda copia del notebook su un runtime solo CPU e dividere il lavoro sulla stessa cartella di Drive:

- runtime GPU: `!python parallel_run.py --role gpu --out-dir /content/drive/MyDrive/xai_v2`
- runtime CPU: `!python parallel_run.py --role flow --out-dir /content/drive/MyDrive/xai_v2`

Quando hanno finito tutti e due, la cella seguente ricostruisce le tabelle complete.

In [ ]:
!python report.py --summarize-only --out-dir /content/drive/MyDrive/xai_v2